In [52]:
# Copyright 2020 , Gurobi Optimization , LLC
# Solve a multi - commodity flow problem . Two products ('Methane'and'Hydrogen')
# are produced in 2 countries ('Denmark'and'Poland') and must be sent to
# consumption in 3 countries ('Germany','Norway', and'Netherlands') to
# satisfy demand ('inflow [h , i ]').
#
# Flows on the transportation network must respect arc capacity constraints
# ('capacity [i , j ]'). The objective is to minimize the sum of the arc
# transportation costs ('cost [i , j ]').
import gurobipy as gp
from gurobipy import GRB
# Base data
commodities = ['Methane' ,'Hydrogen']
nodes = ['Denmark','Poland','Germany','Norway','Netherlands']
arc_capacity_data = {
    ('Denmark', 'Germany', 'Methane'): 100,
    ('Denmark', 'Germany', 'Hydrogen'): 150,
    ('Denmark', 'Norway', 'Methane'): 80,
    ('Denmark', 'Norway', 'Hydrogen'): 120,
    ('Denmark', 'Netherlands', 'Methane'): 120,
    ('Denmark', 'Netherlands', 'Hydrogen'): 180,
    ('Poland', 'Germany', 'Methane'): 120,
    ('Poland', 'Germany', 'Hydrogen'): 180,
    ('Poland', 'Norway', 'Methane'): 120,
    ('Poland', 'Norway', 'Hydrogen'): 150,
    ('Poland', 'Netherlands', 'Methane'): 120,
    ('Poland', 'Netherlands', 'Hydrogen'): 160
}

arcs, capacity = gp.multidict(arc_capacity_data)
                             
    # Cost for triplets commodity - source - destination
cost = {
    ('Methane','Denmark','Germany'): 10 ,
#    ('Methane','Germany','Denmark'): 10 , #added for enable flow in other direction
    ('Methane','Denmark','Norway'): 20 ,
    ('Methane','Denmark','Netherlands'): 60 ,
    ('Methane','Poland','Germany'): 40 ,
    ('Methane','Poland','Norway'): 40 ,
    ('Methane','Poland','Netherlands'): 30 ,
    ('Hydrogen','Denmark','Germany'): 20 ,
    ('Hydrogen','Denmark','Norway'): 20 ,
    ('Hydrogen','Denmark','Netherlands'): 80 ,
    ('Hydrogen','Poland','Germany'): 60 ,
    ('Hydrogen','Poland','Norway'): 70 ,
    ('Hydrogen','Poland','Netherlands'): 30}
# Demand for pairs of commodity - city
inflow = {
    ('Methane','Denmark'): 50 ,
    ('Methane','Poland'): 0 ,
    ('Methane','Germany'): -50 ,
    ('Methane','Norway'): 0 ,
    ('Methane','Netherlands'): 0 ,
    ('Hydrogen','Denmark'): 60 ,
    ('Hydrogen','Poland'): 40 ,
    ('Hydrogen','Germany'): -40 ,
    ('Hydrogen','Norway'): -30 ,
    ('Hydrogen','Netherlands'): -30}
# Create optimization model
m = gp . Model ('netflow')
# Create variables
flow = m . addVars ( commodities , arcs , obj = cost , name = " flow " )

# Arc-capacity constraints with commodity-specific capacities
m.addConstrs(
    (flow.sum('*', i, j) <= capacity[i, j, h] for i, j, h in arcs), "cap"
)

# Flow - conservation constraints
m . addConstrs (
    ( flow . sum (h ,'*', j ) + inflow [h , j ] == flow . sum (h , j ,'*') for h in commodities for j in nodes ) , " node " )


# for h in commodities for j in nodes ) , " node ")

# Compute optimal solution
m . optimize ()
# Print solution
if m.status == GRB.OPTIMAL:
    solution = m.getAttr('x', flow)
    for h in commodities:
        print('\nOptimal flows for %s:' % h)
        for i, j, k in arcs:
            if solution[h, i, j, k] > 0:
                print('%s -> %s (Commodity: %s): %g' % (i, j, k, solution[h, i, j, k]))

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11.0 (22621.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 22 rows, 24 columns and 96 nonzeros
Model fingerprint: 0x891a7137
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+01, 2e+02]
Presolve removed 22 rows and 24 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  0.000000000e+00

Optimal flows for Methane:
Denmark -> Germany (Commodity: Hydrogen): 50

Optimal flows for Hydrogen:
Denmark -> Norway (Commodity: Methane): 30
Denmark -> Netherlands (Commodity: Metha

In [53]:
import pandas as pd

# Print solution
if m.status == GRB.OPTIMAL:
    solution = m.getAttr('x', flow)
    
    # Create a list to store the data
    data = []

    for h in commodities:
        for i, j, k in arcs:
            if solution[h, i, j, k] > 0:
                data.append({
                    'Commodity': h,
                    'Source': i,
                    'Target': j,
                    'Value': solution[h, i, j, k]
                })

    # Create a DataFrame from the collected data
    solution_df = pd.DataFrame(data)

# Print the DataFrame
solution_df

,Commodity,Source,Target,Value
0,Methane,Denmark,Germany,50.0
1,Hydrogen,Denmark,Norway,30.0
2,Hydrogen,Denmark,Netherlands,30.0
3,Hydrogen,Poland,Germany,40.0


In [56]:
import gurobipy as gp
from gurobipy import GRB

# Base data
commodities = ['Methane', 'Hydrogen']
nodes = ['Denmark', 'Poland', 'Germany', 'Norway', 'Netherlands']
arc_capacity_data = {
    ('Denmark', 'Germany', 'Methane'): 100,
    ('Denmark', 'Germany', 'Hydrogen'): 150,
    ('Denmark', 'Norway', 'Methane'): 80,
    ('Denmark', 'Norway', 'Hydrogen'): 120,
    ('Denmark', 'Netherlands', 'Methane'): 120,
    ('Denmark', 'Netherlands', 'Hydrogen'): 180,
    ('Poland', 'Germany', 'Methane'): 120,
    ('Poland', 'Germany', 'Hydrogen'): 180,
    ('Poland', 'Norway', 'Methane'): 120,
    ('Poland', 'Norway', 'Hydrogen'): 150,
    ('Poland', 'Netherlands', 'Methane'): 120,
    ('Poland', 'Netherlands', 'Hydrogen'): 160
}

# Reverse arcs
reverse_arcs = {(j, i, k): cap for (i, j, k), cap in arc_capacity_data.items()}

arcs, capacity = gp.multidict({**arc_capacity_data, **reverse_arcs})

# Cost for triplets commodity - source - destination
cost = {
    ('Methane', 'Denmark', 'Germany'): 10,
    ('Methane', 'Germany', 'Denmark'): 10,  # Enable flow in both directions
    ('Methane', 'Denmark', 'Norway'): 20,
    ('Methane', 'Denmark', 'Netherlands'): 60,
    ('Methane', 'Poland', 'Germany'): 40,
    ('Methane', 'Poland', 'Norway'): 40,
    ('Methane', 'Poland', 'Netherlands'): 30,
    ('Hydrogen', 'Denmark', 'Germany'): 20,
    ('Hydrogen', 'Germany', 'Denmark'): 20,  # Enable flow in both directions
    ('Hydrogen', 'Denmark', 'Norway'): 20,
    ('Hydrogen', 'Denmark', 'Netherlands'): 80,
    ('Hydrogen', 'Poland', 'Germany'): 60,
    ('Hydrogen', 'Poland', 'Norway'): 70,
    ('Hydrogen', 'Poland', 'Netherlands'): 30
}

# Demand for pairs of commodity - city
inflow = {
    ('Methane', 'Denmark'): -50,
    ('Methane', 'Poland'): 0,
    ('Methane', 'Germany'): 50,
    ('Methane', 'Norway'): 0,
    ('Methane', 'Netherlands'): 0,
    ('Hydrogen', 'Denmark'): 60,
    ('Hydrogen', 'Poland'): 40,
    ('Hydrogen', 'Germany'): -40,
    ('Hydrogen', 'Norway'): -30,
    ('Hydrogen', 'Netherlands'): -30
}

# Create optimization model
m = gp.Model('netflow')

# Create variables
flow = m.addVars(commodities, arcs, obj=cost, name="flow")

# Arc-capacity constraints with commodity-specific capacities
m.addConstrs(
    (flow.sum('*', i, j, k) <= capacity[i, j, k] for i, j, k in arcs), "cap"
)

# Flow - conservation constraints
m.addConstrs(
    (flow.sum(h, '*', j, k) + inflow[h, j] == flow.sum(h, j, '*', k) for h in commodities for j in nodes for k in nodes),
    "node"
)

# Compute optimal solution
m.optimize()

# Print solution
if m.status == GRB.OPTIMAL:
    solution = m.getAttr('x', flow)
    for h in commodities:
        print('\nOptimal flows for %s:' % h)
        for i, j, k in arcs:
            if solution[h, i, j, k] > 0:
                print('%s -> %s (Commodity: %s): %g' % (i, j, k, solution[h, i, j, k]))


Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11.0 (22621.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 74 rows, 48 columns and 48 nonzeros
Model fingerprint: 0xe934c21b
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+01, 2e+02]
Presolve time: 0.01s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Infeasible model


In [55]:
reverse_arcs

{('Germany', 'Denmark', 'Methane'): 100,
 ('Germany', 'Denmark', 'Hydrogen'): 150,
 ('Norway', 'Denmark', 'Methane'): 80,
 ('Norway', 'Denmark', 'Hydrogen'): 120,
 ('Netherlands', 'Denmark', 'Methane'): 120,
 ('Netherlands', 'Denmark', 'Hydrogen'): 180,
 ('Germany', 'Poland', 'Methane'): 120,
 ('Germany', 'Poland', 'Hydrogen'): 180,
 ('Norway', 'Poland', 'Methane'): 120,
 ('Norway', 'Poland', 'Hydrogen'): 150,
 ('Netherlands', 'Poland', 'Methane'): 120,
 ('Netherlands', 'Poland', 'Hydrogen'): 160}